# Reasoning Memory: Remember WHY the Agent Decided, Not Just What It Knows

Demos 01-05 store *what* the agent knows. This one stores *why it decided*: the
question, the tool calls, and the sources each decision touched. Without that record,
"why did you recommend that flight?" a week later gets a confident, made-up answer,
because the real reasoning was never kept.

We build it two ways, both live (the agent takes real decisions, nothing hardcoded):

- **Flat** (`agent.state`): a Strands `HookProvider` records each decision's trace and
  it survives a restart via a session manager. Great for replaying one decision.
- **Graph** (Neo4j): the SAME live decisions recorded with Neo4j's official
  `neo4j-agent-memory` SDK, which makes the **reverse audit** possible: "this source
  turned out wrong, which decisions touched it?"

> **What is "flat" memory?** A store that keeps each record on its own (a blob per key,
> or vectors by similarity) with no edges between records. You look a record up, but you
> cannot traverse from one to another. Key-value, a log, and a vector store are all flat
> for this purpose. As Neo4j puts it: *a flat log records what happened; a graph records
> why*. That is the contrast this demo measures.

> **Honesty note:** "reasoning memory" is an engineering pattern, not an established
> academic memory category. What the research supports is the value of traceability and
> provenance ([MemWeaver](https://arxiv.org/abs/2601.18204), the
> [Engram system](https://arxiv.org/abs/2606.09900)). This demo shows the pattern with
> Strands + Neo4j; it is a demo, not production code.


## Why Strands Agents for this demo?

Strands is a good fit because of two framework features, not marketing:

1. **Hooks.** Strands emits lifecycle events (`BeforeInvocationEvent`,
   `AfterToolCallEvent`, `AfterInvocationEvent`) that already carry the data:
   `AfterToolCallEvent` exposes `event.tool_use` (tool name + input). A `HookProvider`
   subscribes to them and records the decision with **zero changes to the tools**.
2. **`agent.state` + session managers.** The trace lives in state, so it persists and
   survives a restart, like any other agent state.

**Why it connects perfectly with Neo4j.** The Strands event gives exactly what Neo4j's
official [`neo4j-agent-memory`](https://neo4j.com/labs/agent-memory/how-to/reasoning-traces/)
SDK needs to record a reasoning trace (`start_trace`, `record_tool_call`,
`complete_trace`). Strands captures *what happened*; Neo4j stores it as a connected,
queryable graph. Neo4j Labs even ships an official Strands integration, so the two are a
natural pair: the hook feeds the SDK with no glue code.


## Prerequisites

1. `OPENAI_API_KEY` (agent model, gpt-4o-mini) and `DUFFEL_API_KEY` (free sandbox token, real flights).
2. For the graph track: a running Neo4j with `NEO4J_*` in `.env`.

```bash
uv venv && uv pip install -r requirements.txt
cp .env.example .env
```


## Install dependencies


In [ ]:
%pip install -q -r requirements.txt


## Configure your model provider

OpenAI by default; switch to Amazon Bedrock, Anthropic, or any Strands provider in the
model cell (see [model providers](https://strandsagents.com/docs/user-guide/concepts/model-providers/?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el)).


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

assert os.getenv('OPENAI_API_KEY'), 'Set OPENAI_API_KEY in .env'
assert os.getenv('DUFFEL_API_KEY'), 'Set DUFFEL_API_KEY in .env (free at https://app.duffel.com)'
print('Provider configured')


## Create the model

One model object, reused by every agent in this notebook.


In [ ]:
os.environ['OTEL_SDK_DISABLED'] = 'true'  # silence tracing noise in notebook output

from strands.models.openai import OpenAIModel
MODEL = OpenAIModel(model_id='gpt-4o-mini')

# Amazon Bedrock instead (no OpenAI key; uses your AWS credentials):
# from strands.models import BedrockModel
# MODEL = BedrockModel(model_id='openai.gpt-oss-120b-1:0', region_name='us-west-2')

SYSTEM_PROMPT = 'You are a travel assistant. Use your tools to answer. Be concise: 2-3 sentences.'
print('Model ready')


---
## The problem: without a trace, the agent confabulates

The agent decides in one session, then we restart it with Strands' own session
management: a new `Agent` instance restores the SAME session from storage (not a
hand-made copy). Asked "why?" after the restart, it has no recorded reasoning, so the
answer is a plausible reconstruction. The session carried the conversation across the
restart, but not the tool-by-tool reasoning. That is the gap the recorder fills.


In [ ]:
from strands import Agent
from strands.session import SnapshotSessionManager
from strands.storage import LocalFileStorage
import tempfile
import trace_kv as kv

SESSION_DIR = tempfile.mkdtemp(prefix='reasoning-sessions-')
def session_manager(session_id):
    return SnapshotSessionManager(session_id=session_id, storage=LocalFileStorage(SESSION_DIR))

# Session 1: decide, NO recorder attached.
deciding = Agent(model=MODEL, system_prompt=SYSTEM_PROMPT,
                 tools=[kv.search_flights, kv.check_fare_alert],
                 session_manager=session_manager('no-trace'), callback_handler=None)
print('Decision:', str(deciding('Find me a flight JFK to Madrid on 2026-10-15 and pick the best.')).strip()[:160])


In [ ]:
# Restart: a fresh Agent restores the same session. Ask WHY.
restarted = Agent(model=MODEL, system_prompt=SYSTEM_PROMPT,
                  tools=[kv.search_flights, kv.check_fare_alert],
                  session_manager=session_manager('no-trace'), callback_handler=None)
print('After restart, asked WHY:', str(restarted('Why did you recommend that Madrid flight?')).strip()[:220])

steps = len(restarted.state.get(kv.TRACES_KEY) or [])
print('\nReal reasoning steps recoverable:', steps)
print('No recorder ran, so the answer above is a reconstruction, not the real chain.')


---
## The recorder: a Strands HookProvider, zero tool changes

`DecisionTraceRecorder` subscribes to three lifecycle events Strands already emits and
assembles a trace into `agent.state`. The tools are never modified. It is written out in
full here because it is the point of the demo.


In [ ]:
from strands.hooks import (
    BeforeInvocationEvent, AfterToolCallEvent, AfterInvocationEvent,
    HookProvider, HookRegistry,
)
TRACES_KEY = 'decision_traces'

class DecisionTraceRecorder(HookProvider):
    """Records one decision trace per invocation into agent.state. No tool changes:
    it reads the events every tool call already emits."""
    def __init__(self): self._current = None
    def register_hooks(self, registry: HookRegistry, **kwargs) -> None:
        registry.add_callback(BeforeInvocationEvent, self._on_start)
        registry.add_callback(AfterToolCallEvent, self._on_tool)
        registry.add_callback(AfterInvocationEvent, self._on_end)
    def _on_start(self, event):
        self._current = {'question': kv._last_user_text(event.messages), 'steps': []}
    def _on_tool(self, event):
        if self._current is None: return
        self._current['steps'].append({
            'n': len(self._current['steps']) + 1,
            'tool': event.tool_use['name'],                 # from Strands' AfterToolCallEvent
            'input': event.tool_use.get('input') or {},
            'evidence': {'name': f"{event.tool_use['name']}-result",
                         'text': kv._result_text(event.result), 'source': event.tool_use['name']},
        })
    def _on_end(self, event):
        if self._current is None: return
        traces = event.agent.state.get(TRACES_KEY) or []
        self._current['id'] = f'trace-{len(traces)+1}'
        self._current['outcome'] = str(event.result).strip()
        traces.append(self._current)
        event.agent.state.set(TRACES_KEY, traces)
        self._current = None

print('DecisionTraceRecorder defined (a HookProvider, no tool changes)')


In [ ]:
# Same tools + hooks=[DecisionTraceRecorder()]. Decide, then restart and replay the REAL chain.
agent = Agent(model=MODEL, system_prompt=SYSTEM_PROMPT,
              tools=[kv.search_flights, kv.check_fare_alert, kv.why_did_i],
              hooks=[kv.DecisionTraceRecorder()],
              session_manager=session_manager('recorder'), callback_handler=None)
print('Decision:', str(agent('Find me a flight JFK to Madrid on 2026-10-15 and pick the best.')).strip()[:140])
traces = agent.state.get(kv.TRACES_KEY) or []
print(f'Recorded automatically: {len(traces)} trace(s), {sum(len(t["steps"]) for t in traces)} step(s)')
for step in traces[0]['steps']:
    print(f"  step {step['n']}: {step['tool']}({step['input']}) -> {step['evidence']['text'][:60]}")


In [ ]:
# Restart: the trace survives in agent.state, so the replay recovers the real chain.
restarted = Agent(model=MODEL, system_prompt=SYSTEM_PROMPT,
                  tools=[kv.search_flights, kv.check_fare_alert, kv.why_did_i],
                  hooks=[kv.DecisionTraceRecorder()],
                  session_manager=session_manager('recorder'), callback_handler=None)
print('After restart, asked WHY:', str(restarted('Why did you recommend that Madrid flight?')).strip()[:220])
trace = kv.replay_why(restarted.state.get(kv.TRACES_KEY) or [], 'Madrid')
print('\nReal reasoning steps recoverable after restart:', len(trace['steps']) if trace else 0)


---
## The graph track: record live decisions with Neo4j's official SDK

Now the same idea on a graph, and this is where the reverse audit becomes possible. We
use Neo4j's official [`neo4j-agent-memory`](https://neo4j.com/labs/agent-memory/how-to/reasoning-traces/)
SDK: we do **not** hand-roll the schema. A Strands `HookProvider`
(`Neo4jDecisionRecorder`, in `trace_graph.py`) records each live decision straight into
Neo4j as the agent runs. The agent takes real decisions with real tools; nothing is
hardcoded.

Neo4j manages this schema for us:

```
(:ReasoningTrace)-[:HAS_STEP]->(:ReasoningStep)-[:USES_TOOL]->(:ToolCall)
(:ReasoningStep)-[:TOUCHED]->(:Entity)
```


In [ ]:
import trace_graph as tg

# A clean, isolated Neo4j database for this demo (the SDK creates its own indexes there).
tg.ensure_clean_database()
print('reasoningdemo database ready and clean')


### The agent takes ten real decisions

A travel-planning session: flights, a budget, an itinerary, insurance, a calendar block,
weather, dining. Each is a real invocation; the recorder captures whatever tools the
agent actually called. Some read the fare-alerts feed, some read weather or the dining
guide. That mix is what the reverse audit will sort out.


In [ ]:
PROMPTS = [
    'Find me a flight from JFK to Madrid on 2026-10-15 and pick the best option.',
    'Now find me a flight from JFK to Tokyo on 2026-10-20 and pick the best.',
    'Is there a fare alert on the JFK to Madrid route right now?',
    'What is the best time of year to visit Tokyo?',
    'What should I pack for Madrid in October?',
    'Check the fare alert for JFK to London.',
    'When should I visit Madrid for good weather?',
    'Find me a flight from JFK to Paris on 2026-11-01 and pick the best.',
    'Any fare alert on JFK to Paris?',
    'What is the weather like in Tokyo in October?',
]

agent = Agent(model=MODEL, system_prompt=SYSTEM_PROMPT,
              tools=[kv.search_flights, kv.check_fare_alert, kv.best_time_to_visit, kv.check_weather],
              hooks=[tg.Neo4jDecisionRecorder(session_id='travel')],
              callback_handler=None)
for p in PROMPTS:
    resp = agent(p)
    print(f'- {p[:55]:55} -> {str(resp).strip()[:60]}')
print('\nAll decisions recorded live into Neo4j.')


### Replay: why did I decide that? (the real chain, no model call)


In [ ]:
import asyncio
async def _replay():
    async with tg.memory_client() as client:
        r = await tg.replay_why(client, 'flight from JFK to Madrid')
        print('Q:', r['question'])
        print('Outcome:', r['outcome'][:120])
        print('Tools it actually used:', r['tools'])
await _replay()


### The reverse audit: "the fare-alerts feed was compromised, which decisions touched it?"

This is the question a flat store cannot answer with one query. In the graph it is a
single traversal over the `:TOUCHED` edges the SDK recorded. The decisions that only
read weather or the dining guide are correctly left out.


In [ ]:
async def _audit():
    async with tg.memory_client() as client:
        total = await tg.all_decisions(client)
        affected = await tg.find_affected(client)   # touched fare_alerts_feed
        print(f'Total decisions recorded: {len(total)}')
        print(f'Touched the compromised fare_alerts_feed: {len(affected)}')
        for a in affected: print('  -', a[:60])
        print('\nSources each decision touched:')
        for row in await tg.sources_touched(client):
            print(f"  {row['decision'][:50]:50} -> {row['sources']}")
await _audit()


### See the graph yourself (Neo4j Browser)

Open Neo4j Browser, select the `reasoningdemo` database, and run one of these. They
return paths, so the Browser draws the edges (if you only return nodes, turn on
"Connect result nodes" in the Browser settings).


In [ ]:
for name, query in tg.VISUALIZE_QUERIES.items():
    print(f'# --- {name} ---')
    print(query)
    print()


![The reasoning graph in Neo4j: decisions, their tool calls, and the sources each touched](images/ai-agent-reasoning-graph-neo4j.png)

*Replace with a screenshot of the `full_graph` query in Neo4j Browser.*


---
## Why store the reasoning at all? Tokens saved, errors avoided

Replaying "why did I decide X?" from the stored trace is a read: **0 model tokens**, and
it returns the real chain. Asking the model to reconstruct it with no trace costs tokens
*and* confabulates. Storing the trace saves tokens and avoids errors.


In [ ]:
# (a) Replay from the stored trace: no model call.
async def _cost():
    async with tg.memory_client() as client:
        r = await tg.replay_why(client, 'flight from JFK to Madrid')
    print('(a) Replay from the trace: 0 model tokens, the real recorded chain:', r['tools'])
await _cost()

# (b) Ask the model to reconstruct it, with no trace.
reconstructor = Agent(model=MODEL, system_prompt=SYSTEM_PROMPT, callback_handler=None)
resp = reconstructor('Earlier you recommended a JFK-Madrid flight. Reconstruct the exact tool-by-tool reasoning you used.')
print('(b) Reconstruct with the model:', resp.metrics.accumulated_usage['totalTokens'],
      'tokens, and the chain is confabulated (no trace existed).')


---
## A word on privacy: showing the reasoning is not always safe

This is a demo. In production, replaying a decision's reasoning on request can leak
information: a trace can contain another user's data, internal sources, or PII the asker
should not see. Some ways to keep the replay safe, reusing patterns from this repo:

- **Screen what you record and what you replay.** The [memory-hygiene demo](../05-memory-hygiene-demo/)
  ships a `screen_memory` write-gate (regex for injection and PII shapes, plus an LLM
  classifier). Run traces through it before storing, and before returning them.
- **Detect PII at the boundary.** As the [caching sample](https://github.com/elizabethfuentes12/stop-paying-for-repeated-llm-calls-sample-for-aws)
  recommends, run content through [Amazon Comprehend PII detection](https://docs.aws.amazon.com/comprehend/latest/dg/how-pii.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el)
  and redact or skip anything flagged.
- **Scope by tenant and role.** Partition traces per user/tenant and check the asker is
  entitled to a decision before replaying it. Neo4j's SDK documents a
  [privacy-and-audit](https://neo4j.com/labs/agent-memory/how-to/privacy-and-audit/) path
  for this.
- **Return the outcome, redact the internals.** A reviewer may need "which source did
  this touch?" without the raw evidence text.


---
## Optional: the same hooks can *reuse* reasoning, not just audit it

Auditing looks backward: record the decision, traverse it later. The same
`HookProvider` mechanism can look forward and **reuse** a past plan, which is a
*reasoning cache*: on a new-but-similar question, inject a matching past tool plan and
skip the exploration, saving tokens and latency.

That is a different concern (reuse, not audit), and it is built in a companion sample:
[Prompt Caching Isn't Enough: Semantic and Reasoning Caches for AI Agents](https://github.com/elizabethfuentes12/stop-paying-for-repeated-llm-calls-sample-for-aws).
There a `ReasoningCache` is a Strands `HookProvider` too: `BeforeInvocationEvent` injects
a matching past tool plan, `AfterInvocationEvent` stores this question's trajectory as its
plan. Same events as our recorder here, opposite direction: this demo *records to audit*,
that one *reads to reuse*. AWS's published benchmark for semantic caching reports up to
86% cost and 88% latency reduction.


---
## Cleanup

Drop the isolated `reasoningdemo` database, so nothing this demo created is left behind.
Guarded so it never touches other databases.


In [ ]:
import trace_graph as tg
tg.teardown_database()
print('Teardown complete: reasoningdemo dropped.')
